# ECQL: LoRA против базовой модели

Перевод вопроса на русском в запрос на внутреннем языке ECQL.

Ноутбук работает в двух местах:
- **Google Colab, T4** - база грузится в 4 битах через bitsandbytes;
- **Mac, Apple Silicon** - bitsandbytes не работает, база грузится в bf16.

Порядок: 
- самопроверка метрик, 
- два прогона базовой модели, 
- обучение адаптера,
- прогон с адаптером, 
- сравнение.

## 1. Окружение

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q \
        transformers==5.15.1 \
        peft==0.20.0 \
        trl==1.10.0 \
        accelerate==1.14.0 \
        datasets==5.0.1 \
        nltk==3.10.3 \
        bitsandbytes

from importlib.metadata import PackageNotFoundError, version

for package in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "nltk", "bitsandbytes"):
    try:
        print(f"{package:16} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:16} не установлен")

## 2. Код и данные

Код лежит в репозитории: `src/ecql_dataset`. 

В Colab репозиторий клонируется, локально берётся из каталога, где лежит ноутбук.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/samtakoy/llm-engineer-ecql-and-metrcis.git"

if IN_COLAB:
    ROOT = Path("/content") / Path(REPO_URL).stem
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))

DATASET = ROOT / "dataset" / "ecql"
print("корень:", ROOT)
print("датасет:", DATASET)

In [ ]:
import json

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]

train = read_jsonl(DATASET / "train.jsonl")
val = read_jsonl(DATASET / "val.jsonl")
test = read_jsonl(DATASET / "test.jsonl")


print(f"train {len(train)}, val {len(val)}, test {len(test)}, "
      f"из них challenge {sum(r['meta']['challenge'] for r in test)}")

print()
print(test[0]["input"])
print(test[0]["output"])

## 3. Устройство

Отсюда берутся размер батча и способ загрузки модели.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.bfloat16
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1

print({"устройство": DEVICE, "тип": str(DTYPE), "4 бита": LOAD_IN_4BIT, "батч": BATCH_SIZE})

## 4. Самопроверка метрик

Метрике подаются эталоны вместо ответов модели. Идеальный прогон обязан дать единицу по синтаксису, логике и строке и ноль галлюцинаций.

Проверка идёт до запуска модели: сломанная метрика делает бессмысленными все последующие цифры.

In [ ]:
from ecql_dataset.notebook.eval.product import judge, self_check

print(self_check(records=test))

- ответов 68 — весь тест;
- синтаксис 1.0 — все 68 разобрались по грамматике;
- логика 1.0 и пять частей по 1.0 — каждый совпал с собой;
- строка 1.0 — посимвольно тоже;
- галлюцинации 0.0 — чужого языка и выдуманных полей нет.

### Проверка на испорченных ответах

Самопроверка показала, что метрика не занижает. Теперь посмотрим — что не завышает.

Шесть ответов, каждый сломан по-своему. Логика должна упасть везде, кроме перестановки условий: там порядок другой, а смысл тот же.

In [ ]:
reference = "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700 AS LIST"
cases = {
    "переставлены условия": "FETCH [PLACES] WHERE @price_rub BELOW 700 && @category IS 'food' AS LIST",
    "испорчено значение": "FETCH [PLACES] WHERE @category IS 'culture' && @price_rub BELOW 700 AS LIST",
    "перепутан оператор": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub ABOVE 700 AS LIST",
    "забыт суффикс": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700",
    "SQL вместо ECQL": "SELECT * FROM places WHERE category = 'food'",
    "выдуманное поле": "FETCH [PLACES] WHERE @cuisine IS 'food' && @price_rub BELOW 700 AS LIST",
}

for name, prediction in cases.items():
    verdict = judge(prediction=prediction, reference=reference)
    print(f"{name:22} строка={int(verdict.exact)} логика={int(verdict.logic)} "
          f"галлюцинации={int(verdict.hallucination)}  {verdict.reason}")

**Проверка: Текстовая метрика не заменяет логическую**

В каждом эталоне портится одно значение. Запрос остаётся правильным по
синтаксису и почти совпадает с эталоном как текст, но возвращает не те данные.

In [ ]:
import re

from ecql_dataset.notebook.eval import text as text_metrics
from ecql_dataset.notebook.eval.product import evaluate

references = [record["output"] for record in test]
broken = [re.sub(r"'[^']*'", "'сломано'", reference, count=1) for reference in references]

rows = {}
for name, predictions in (("эталоны", references), ("одно значение испорчено", broken)):
    product, _ = evaluate(records=test, predictions=predictions)
    rows[name] = text_metrics.score(predictions=predictions, references=references) | {
        "логика": product["логика"],
        "синтаксис": product["синтаксис"],
    }

names = ["exact", "token_f1", "bleu", "meteor", "rouge_l", "cider", "синтаксис", "логика"]
print(f"{'метрика':12} {'эталоны':>10} {'испорчено':>12}")
for name in names:
    print(f"{name:12} {rows['эталоны'][name]:>10.3f} {rows['одно значение испорчено'][name]:>12.3f}")

Испорченный запрос синтаксически правильный и текстово почти совпадает с эталоном, но возвращает другие данные.

Все текстовые метрики, кроме exact, показывают 0.6–0.94 — «почти правильно». Логика показывает 0.

exact строгая, но слепая к перестановке условий.

Значит ориентироваться на них нельзя — подмену значения они не видят. 

Текстовые метрики отвечают «стало ли похоже на язык», логическая — «правильный ли запрос».